In [1]:
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

In [2]:
spark.sparkContext.setLogLevel("ERROR")  # or "WARN"
spark

In [3]:
%%sql
SHOW CATALOGS

catalog
demo
spark_catalog


In [14]:
!ls /home/iceberg/data_sync/

customer.parquet  nation.parquet  partsupp.parquet  supplier.parquet
game.csv	  orders.parquet  pokemon.csv
lineitem.parquet  part.parquet	  region.parquet


In [15]:
%%sql
CREATE DATABASE IF NOT EXISTS demoTryo

++
||
++
++

In [16]:
%%sql
SHOW DATABASES

namespace
demoTryo


In [19]:
df_nba = spark.read.csv("/home/iceberg/data_sync/game.csv", header=True, inferSchema=True)

In [21]:
df_nba.printSchema()

root
 |-- season_id: integer (nullable = true)
 |-- team_id_home: integer (nullable = true)
 |-- team_abbreviation_home: string (nullable = true)
 |-- team_name_home: string (nullable = true)
 |-- game_id: integer (nullable = true)
 |-- game_date: timestamp (nullable = true)
 |-- matchup_home: string (nullable = true)
 |-- wl_home: string (nullable = true)
 |-- min: integer (nullable = true)
 |-- fgm_home: double (nullable = true)
 |-- fga_home: double (nullable = true)
 |-- fg_pct_home: double (nullable = true)
 |-- fg3m_home: double (nullable = true)
 |-- fg3a_home: double (nullable = true)
 |-- fg3_pct_home: double (nullable = true)
 |-- ftm_home: double (nullable = true)
 |-- fta_home: double (nullable = true)
 |-- ft_pct_home: double (nullable = true)
 |-- oreb_home: double (nullable = true)
 |-- dreb_home: double (nullable = true)
 |-- reb_home: double (nullable = true)
 |-- ast_home: double (nullable = true)
 |-- stl_home: double (nullable = true)
 |-- blk_home: double (nullable

In [23]:
df_nba.writeTo("demoTryo.nba") \
  .using("iceberg") \
  .partitionedBy(f.col("season_id")) \
  .createOrReplace()

In [24]:
%%sql
SELECT 
    *
FROM demoTryo.nba.snapshots

committed_at,snapshot_id,parent_id,operation,manifest_list,summary
2025-11-20 20:37:17.798000,2062004969588677606,None,append,s3://warehouse/demoTryo/nba/metadata/snap-2062004969588677606-1-9b76261e-e506-446a-8a9f-4e33ecfbad34.avro,"{'engine-version': '3.5.5', 'added-data-files': '225', 'total-equality-deletes': '0', 'app-id': 'local-1763422686127', 'added-records': '65698', 'total-records': '65698', 'spark.app.id': 'local-1763422686127', 'changed-partition-count': '225', 'engine-name': 'spark', 'total-position-deletes': '0', 'added-files-size': '7290724', 'total-delete-files': '0', 'iceberg-version': 'Apache Iceberg 1.8.1 (commit 9ce0fcf0af7becf25ad9fc996c3bad2afdcfd33d)', 'total-files-size': '7290724', 'total-data-files': '225'}"


In [25]:
%%sql
SELECT *
FROM   demoTryo.nba.manifests

content,path,length,partition_spec_id,added_snapshot_id,added_data_files_count,existing_data_files_count,deleted_data_files_count,added_delete_files_count,existing_delete_files_count,deleted_delete_files_count,partition_summaries
0,s3://warehouse/demoTryo/nba/metadata/9b76261e-e506-446a-8a9f-4e33ecfbad34-m0.avro,110464,0,2062004969588677606,225,0,0,0,0,0,"[Row(contains_null=False, contains_nan=False, lower_bound='12005', upper_bound='42022')]"


In [26]:
%%sql
SELECT * from demoTryo.nba.partitions

partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
Row(season_id=12008),0,110,1,30072,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=31976),0,2,1,17656,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=12009),0,118,1,30715,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=31977),0,2,1,17871,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=12006),0,120,1,30901,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=31974),0,2,1,17655,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=12007),0,106,1,30390,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=31975),0,1,1,15637,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=12012),0,116,1,30673,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
Row(season_id=31980),0,1,1,16150,0,0,0,0,2025-11-20 20:37:17.798000,2062004969588677606
